# EDA — Triagem de Doenças Oculares em Imagens de Retina

Antes de rodar: baixe o dataset do Kaggle e organize em `data/raw/dataset/<classe>/` (ver README).

In [ ]:
import sys
sys.path.append("../src")

import matplotlib.pyplot as plt
import tensorflow as tf

from load_data import contar_imagens_por_classe, carregar_datasets, DATASET_DIR

## 1. Quantas imagens por classe?

O dataset é anunciado como praticamente balanceado (~1000 por classe) — vamos confirmar.

In [ ]:
contagem = contar_imagens_por_classe()
contagem

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(contagem.keys(), contagem.values())
plt.title("Imagens por classe")
plt.ylabel("Quantidade")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 2. Ver exemplos de cada classe

Sempre vale olhar as imagens de verdade antes de treinar qualquer coisa — ajuda a notar diferenças de qualidade/iluminação entre classes que podem virar viés (ver docs/notas_eticas.md).

In [ ]:
import random
from pathlib import Path

classes = sorted(contagem.keys())
fig, axs = plt.subplots(len(classes), 4, figsize=(12, 3 * len(classes)))

for i, classe in enumerate(classes):
    pasta = DATASET_DIR / classe
    arquivos = list(pasta.glob("*.*"))
    amostras = random.sample(arquivos, min(4, len(arquivos)))
    for j, caminho in enumerate(amostras):
        img = plt.imread(caminho)
        axs[i, j].imshow(img)
        axs[i, j].axis("off")
        if j == 0:
            axs[i, j].set_ylabel(classe)
    axs[i, 0].set_title(classe, loc="left", fontsize=10)

plt.tight_layout()
plt.show()

## 3. Carregar os datasets (treino/validação/teste)

Usa o mesmo split que o `src/model.py` vai usar no treino — conferimos aqui só para ver o tamanho de cada batch e a ordem das classes.

In [ ]:
treino_ds, val_ds, teste_ds, class_names = carregar_datasets()

print("Classes:", class_names)
print("Batches -> treino:", tf.data.experimental.cardinality(treino_ds).numpy())
print("Batches -> validação:", tf.data.experimental.cardinality(val_ds).numpy())
print("Batches -> teste:", tf.data.experimental.cardinality(teste_ds).numpy())

## 4. Conferir um batch de imagens já carregado

In [ ]:
for imagens, rotulos in treino_ds.take(1):
    fig, axs = plt.subplots(1, 6, figsize=(15, 3))
    for i in range(6):
        axs[i].imshow(imagens[i].numpy().astype("uint8"))
        axs[i].set_title(class_names[rotulos[i]])
        axs[i].axis("off")
    plt.tight_layout()
    plt.show()

## 5. Próximos passos

O treino de fato (transfer learning com MobileNetV2 + fine-tuning) está em `src/model.py`, para rodar fora do notebook:

```bash
python src/model.py
```